In [ ]:
# ============================================
# 1) CellTalkDB data
# ============================================

In [ ]:
import pandas as pd
import re
from pathlib import Path

print("Loading files...")

# UPDATED receptor list path here 👇
RECEPTOR_LIST_XLSX = "/Users/antina/Downloads/STAT3_activating_receptors_mouse_annotated_UPDATED.xlsx"
LR_PAIRS_XLSX = "/Users/antina/Downloads/Endothelium-Endfoot ligand-receptor pairs 010525_BlancaDiazCastro.xlsx"


# Optional: change output name so you don’t overwrite the old one
OUT_XLSX           = "CellTalkDB_pairs_filtered_STAT3_receptors_UPDATED.xlsx"

receptors_df = pd.read_excel(RECEPTOR_LIST_XLSX)
lr_df = pd.read_excel(LR_PAIRS_XLSX)

print("Receptors loaded:", receptors_df.shape)
print("LR pairs loaded:", lr_df.shape)

def split_genes(x):
    if pd.isna(x):
        return []
    parts = re.split(r"[;,/|\s]+", str(x).strip())
    return [p for p in parts if p]

lig_col = "BEC Ligand mouse gene name"
rec_col = "Endfoot Receptor mouse gene name"
lps_col = "Response to LPS (ligand differential expression, receptors don't change)"
pair_col = "Ligand-receptor pair"

receptor_whitelist = set(
    receptors_df["Receptor_gene_symbol_mouse"].astype(str).str.strip()
)

lr_df["rec_list"] = lr_df[rec_col].apply(split_genes)

lr_df["STAT3_receptor_hit"] = lr_df["rec_list"].apply(
    lambda genes: any(g in receptor_whitelist for g in genes)
)

hits = lr_df[lr_df["STAT3_receptor_hit"]].copy()
print("STAT3-hit LR rows (pre-explode):", hits.shape)

# explode so multiple receptors in a cell become separate rows
hits = hits.explode("rec_list").rename(columns={"rec_list": "Matched_Receptor"})

# keep only matched receptors
hits = hits[hits["Matched_Receptor"].isin(receptor_whitelist)].copy()

# add annotations from the receptor list
hits = hits.merge(
    receptors_df,
    left_on="Matched_Receptor",
    right_on="Receptor_gene_symbol_mouse",
    how="left"
)

out_cols = [
    pair_col,
    lig_col,
    rec_col,
    "Matched_Receptor",
    "STAT3_activation_type",
    "Source",
    lps_col
]
out_cols = [c for c in out_cols if c in hits.columns]

hits_out = hits[out_cols].copy()

Path(OUT_XLSX).parent.mkdir(parents=True, exist_ok=True)
hits_out.to_excel(OUT_XLSX, index=False)

print("Saved to:", OUT_XLSX)
print("Final rows kept:", hits_out.shape[0])

hits_out.head(), hits_out.shape


In [ ]:
# ============================================
# 2) ASTRO VS ENDFEET + editable charts
# ============================================

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION
from pptx.chart.data import CategoryChartData, XyChartData

# ============================================
# FILE PATHS
# ============================================
ENDFEET_FILE = "/Users/antina/Desktop/NeuroPSI/data mining/enfeetVSastro-proteom-stats.xlsx"
STAT3_FILE = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_full_pathway_plus_receptors_mouse.xlsx"

OUT_TABLE = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_2.xlsx"
OUT_VOLCANO = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_volcano.png"
OUT_BARPLOT = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_barplot_2.png"
OUT_VOLCANO_POINTS_ONLY = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_volcano_points_only.png"
OUT_PPTX = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_editable_charts.pptx"

# ============================================
# HELPERS
# ============================================
def first_gene_symbol(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    s = s.split(";")[0]
    s = s.split(" ")[0]
    return s.upper()

def p_to_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""
    
def finite_or_none(x):
    try:
        x = float(x)
        if np.isfinite(x):
            return x
    except Exception:
        pass
    return None
# ============================================
# LOAD EF vs ASTRO COMPARISON SHEET
# ============================================
cmp_sheet = "EF vs Astro (3194)"
df_cmp = pd.read_excel(ENDFEET_FILE, sheet_name=cmp_sheet, header=0)
df_cmp.columns = df_cmp.columns.str.strip()

required_cmp_cols = [
    "Mouse.Gene",
    "Endfoot Avg Protein abundance",
    "Astrocyte Avg Protein abundance",
    "Log2FC (EF vs. Astro)",
    "P.Value"
]

for col in required_cmp_cols:
    if col not in df_cmp.columns:
        raise ValueError(f"Missing expected column in comparison sheet: {col}")

df_cmp["Gene_clean"] = df_cmp["Mouse.Gene"].apply(first_gene_symbol)

# convert numeric columns
num_cols = [
    "Endfoot Avg Protein abundance",
    "Astrocyte Avg Protein abundance",
    "Log2FC (EF vs. Astro)",
    "P.Value"
]
for c in num_cols:
    df_cmp[c] = pd.to_numeric(df_cmp[c], errors="coerce")

print("Comparison sheet loaded:", df_cmp.shape)

# ============================================
# LOAD STAT3 LIST
# ============================================
stat3 = pd.read_excel(STAT3_FILE)
stat3.columns = stat3.columns.str.strip()

possible_gene_cols = [
    "Gene",
    "Gene_symbol",
    "Gene Symbol",
    "Genes",
    "Mouse_gene_symbol",
    "Receptor_gene_symbol_mouse"
]

stat3_gene_col = None
for c in possible_gene_cols:
    if c in stat3.columns:
        stat3_gene_col = c
        break

if stat3_gene_col is None:
    stat3_gene_col = stat3.columns[0]

print("Using STAT3 gene column:", stat3_gene_col)

stat3_genes = set(
    stat3[stat3_gene_col]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

# ============================================
# KEEP ONLY STAT3-RELEVANT PROTEINS
# ============================================
df_stat3 = df_cmp[df_cmp["Gene_clean"].isin(stat3_genes)].copy()
print("STAT3-relevant proteins detected:", df_stat3.shape)

if df_stat3.shape[0] == 0:
    raise ValueError("No STAT3-relevant proteins found in the comparison sheet.")

# ============================================
# PREPARE OUTPUT TABLE
# ============================================
df_stat3["stars"] = df_stat3["P.Value"].apply(p_to_stars)
df_stat3["neglog10_pvalue"] = -np.log10(df_stat3["P.Value"].replace(0, np.nan))

# log2 display values for bar plot readability
df_stat3["Astro avg log2"] = np.log2(df_stat3["Astrocyte Avg Protein abundance"] + 1)
df_stat3["Endfoot avg log2"] = np.log2(df_stat3["Endfoot Avg Protein abundance"] + 1)

# Add enrichment direction
df_stat3["Enriched_in"] = np.where(
    df_stat3["Log2FC (EF vs. Astro)"] > 0, "Endfeet",
    np.where(df_stat3["Log2FC (EF vs. Astro)"] < 0, "Astrocytes", "Equal")
)

keep_cols = [
    "Gene_clean",
    "Mouse.Gene",
    "Human.Gene",
    "Uniprot ID",
    "Protein Description",
    "Clasification",
    "Astrocyte Avg Protein abundance",
    "Endfoot Avg Protein abundance",
    "Astro avg log2",
    "Endfoot avg log2",
    "Log2FC (EF vs. Astro)",
    "P.Value",
    "stars",
    "Enriched_in"
]
keep_cols = [c for c in keep_cols if c in df_stat3.columns] + ["neglog10_pvalue"]

df_stat3 = df_stat3[keep_cols].copy()
df_stat3 = df_stat3.sort_values(["P.Value", "Log2FC (EF vs. Astro)"], ascending=[True, False])

df_stat3.to_excel(OUT_TABLE, index=False)

# ============================================
# VOLCANO PLOT
# ============================================
plt.figure(figsize=(8, 6))

plt.scatter(
    df_stat3["Log2FC (EF vs. Astro)"],
    df_stat3["neglog10_pvalue"],
    s=50
)

for _, row in df_stat3.iterrows():
    label = str(row["Gene_clean"])
    if row["stars"] != "":
        label = f"{label} {row['stars']}"
    plt.text(
        row["Log2FC (EF vs. Astro)"],
        row["neglog10_pvalue"],
        label,
        fontsize=8
    )

plt.axvline(1, linestyle="--", linewidth=1)
plt.axvline(-1, linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), linestyle="--", linewidth=1)

plt.xlabel("Log2FC (Endfeet / Astrocytes)")
plt.ylabel("-log10 P.Value")
plt.title("STAT3-relevant proteins: Endfeet vs Astrocytes")
plt.tight_layout()
plt.savefig(OUT_VOLCANO, dpi=300)
plt.close()

# ============================================
# VOLCANO PLOT (POINTS ONLY - NO LABELS)
# ============================================
plt.figure(figsize=(8, 6))

plt.scatter(
    df_stat3["Log2FC (EF vs. Astro)"],
    df_stat3["neglog10_pvalue"],
    s=50
)

plt.axvline(1, linestyle="--", linewidth=1)
plt.axvline(-1, linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), linestyle="--", linewidth=1)

plt.xlabel("Log2FC (Endfeet / Astrocytes)")
plt.ylabel("-log10 P.Value")
plt.title("STAT3-relevant proteins: Endfeet vs Astrocytes (points only)")
plt.tight_layout()
plt.savefig(OUT_VOLCANO_POINTS_ONLY, dpi=300)
plt.close()

# ============================================
# BAR PLOT
# ============================================
bar_df = df_stat3.sort_values("Endfoot avg log2", ascending=False).reset_index(drop=True)

x = np.arange(len(bar_df))
width = 0.38

plt.figure(figsize=(max(10, len(bar_df) * 0.4), 6))

plt.bar(x - width/2, bar_df["Astro avg log2"], width=width, label="Astrocytes")
plt.bar(x + width/2, bar_df["Endfoot avg log2"], width=width, label="Endfeet")

for i, row in bar_df.iterrows():
    ymax = max(row["Astro avg log2"], row["Endfoot avg log2"])
    if row["stars"] != "":
        plt.text(i, ymax + 0.1, row["stars"], ha="center", va="bottom", fontsize=10)

plt.xticks(x, bar_df["Gene_clean"], rotation=90)
plt.ylabel("Mean log2 protein abundance")
plt.title("STAT3-relevant proteins: Astrocytes vs Endfeet")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_BARPLOT, dpi=300)
plt.close()

# ============================================
# CREATE EDITABLE POWERPOINT
# ============================================
prs = Presentation()
prs.slide_width = Inches(13.33)
prs.slide_height = Inches(7.5)

# ---- Slide 1: Editable volcano scatter ----
slide = prs.slides.add_slide(prs.slide_layouts[6])

title_box = slide.shapes.add_textbox(Inches(0.4), Inches(0.2), Inches(12.0), Inches(0.5))
title_box.text_frame.text = "STAT3-relevant proteins - Endfeet vs Astrocytes - Volcano plot"

# keep only finite values for pptx chart
volcano_ppt = df_stat3.copy()
volcano_ppt["x_ppt"] = volcano_ppt["Log2FC (EF vs. Astro)"].apply(finite_or_none)
volcano_ppt["y_ppt"] = volcano_ppt["neglog10_pvalue"].apply(finite_or_none)
volcano_ppt = volcano_ppt.dropna(subset=["x_ppt", "y_ppt"]).copy()

xy_data = XyChartData()
series = xy_data.add_series("STAT3 proteins")

for _, row in volcano_ppt.iterrows():
    series.add_data_point(float(row["x_ppt"]), float(row["y_ppt"]))

chart = slide.shapes.add_chart(
    XL_CHART_TYPE.XY_SCATTER,
    Inches(0.7), Inches(0.9), Inches(8.7), Inches(5.8),
    xy_data
).chart

chart.has_legend = False
chart.chart_title.has_text_frame = True
chart.chart_title.text_frame.text = "Log2FC vs -log10(P.Value)"

value_axis = chart.value_axis
value_axis.has_title = True
value_axis.axis_title.text_frame.text = "-log10 P.Value"

category_axis = chart.category_axis
category_axis.has_title = True
category_axis.axis_title.text_frame.text = "Log2FC (Endfeet / Astrocytes)"

# editable labels as text boxes
x_min = min(-1.5, float(volcano_ppt["x_ppt"].min()) - 0.3)
x_max = max(1.5, float(volcano_ppt["x_ppt"].max()) + 0.3)
y_min = 0.0
y_max = max(1.5, float(volcano_ppt["y_ppt"].max()) + 0.5)

plot_left = 0.7
plot_top = 0.9
plot_w = 8.7
plot_h = 5.8

for _, row in volcano_ppt.iterrows():
    xval = float(row["x_ppt"])
    yval = float(row["y_ppt"])
    label = str(row["Gene_clean"])
    if row["stars"] != "":
        label = f"{label} {row['stars']}"

    x_norm = (xval - x_min) / (x_max - x_min) if x_max > x_min else 0.5
    y_norm = (yval - y_min) / (y_max - y_min) if y_max > y_min else 0.5

    left = plot_left + x_norm * plot_w
    top = plot_top + (1 - y_norm) * plot_h

    tb = slide.shapes.add_textbox(Inches(left + 0.02), Inches(top - 0.02), Inches(1.2), Inches(0.2))
    p = tb.text_frame.paragraphs[0]
    p.text = label
    p.font.size = Pt(8)

    
# ---- Slide 2: Editable bar chart ----
slide = prs.slides.add_slide(prs.slide_layouts[6])

title_box = slide.shapes.add_textbox(Inches(0.4), Inches(0.2), Inches(12.0), Inches(0.5))
title_box.text_frame.text = "STAT3-relevant proteins - Endfeet vs Astrocytes - Bar plot"

bar_df_ppt = df_stat3.sort_values("Endfoot avg log2", ascending=False).reset_index(drop=True).copy()

# keep only finite values for pptx chart
bar_df_ppt["astro_ppt"] = bar_df_ppt["Astro avg log2"].apply(finite_or_none)
bar_df_ppt["endfoot_ppt"] = bar_df_ppt["Endfoot avg log2"].apply(finite_or_none)
bar_df_ppt = bar_df_ppt.dropna(subset=["astro_ppt", "endfoot_ppt"]).copy()

chart_data = CategoryChartData()
chart_data.categories = list(bar_df_ppt["Gene_clean"])
chart_data.add_series("Astrocytes", [float(v) for v in bar_df_ppt["astro_ppt"]])
chart_data.add_series("Endfeet", [float(v) for v in bar_df_ppt["endfoot_ppt"]])

chart = slide.shapes.add_chart(
    XL_CHART_TYPE.COLUMN_CLUSTERED,
    Inches(0.6), Inches(0.9), Inches(12.0), Inches(5.8),
    chart_data
).chart

chart.has_legend = True
chart.legend.position = XL_LEGEND_POSITION.BOTTOM
chart.legend.include_in_layout = False

chart.value_axis.has_title = True
chart.value_axis.axis_title.text_frame.text = "Mean log2 protein abundance"

# editable significance stars
n = len(bar_df_ppt)
for i, row in bar_df_ppt.iterrows():
    if row["stars"] == "":
        continue

    frac = (i + 0.5) / max(n, 1)
    left = 0.6 + frac * 12.0 - 0.12
    top = 1.0

    tb = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(0.3), Inches(0.2))
    p = tb.text_frame.paragraphs[0]
    p.text = row["stars"]
    p.font.size = Pt(10)

# save the PowerPoint file
prs.save(OUT_PPTX)

print("PPTX saved:", OUT_PPTX)

# ============================================
# SUMMARY
# ============================================
print("\nSaved outputs:")
print(OUT_TABLE)
print(OUT_VOLCANO)
print(OUT_VOLCANO_POINTS_ONLY)
print(OUT_BARPLOT)
print(OUT_PPTX)


print("\nPreview:")
print(df_stat3[[
    c for c in [
        "Gene_clean",
        "Astro avg log2",
        "Endfoot avg log2",
        "Log2FC (EF vs. Astro)",
        "P.Value",
        "stars",
        "Enriched_in"
    ] if c in df_stat3.columns
]].head(20))

Comparison sheet loaded: (3193, 10)
Using STAT3 gene column: Gene
STAT3-relevant proteins detected: (25, 10)
PPTX saved: /Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_editable_charts.pptx

Saved outputs:
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_2.xlsx
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_volcano.png
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_volcano_points_only.png
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_barplot_2.png
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_astro_vs_endfeet_NO_TurboID_filter_editable_charts.pptx

Preview:
     Gene_clean  Astro avg log2  Endfoot avg log2  Log2FC (EF vs. Astro)  \
1415      PTPRA       15.540783         17.137688               1.604089   
1471      PTPRS       16.019329         17.497428               1.483827   
1146      ERBB2       

In [ ]:
# ============================================
# 3) ENDFEET LPS VS PBS + editable charts
# ============================================

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION
from pptx.chart.data import CategoryChartData, XyChartData
from pptx.enum.shapes import MSO_AUTO_SHAPE_TYPE

# ============================================
# FILE PATHS
# ============================================
DATA_FILE = "/Users/antina/Desktop/NeuroPSI/data mining/endfeet-proteom-LPS-PBS-stats.xlsx"
STAT3_FILE = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_full_pathway_plus_receptors_mouse.xlsx"

OUT_TABLE = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_LPSvPBS_filtered.xlsx"
OUT_VOLCANO = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_volcano_pipeline_stats.png"
OUT_BARPLOT = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_barplot_pipeline_stats.png"
OUT_PPTX = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_editable_charts.pptx"
OUT_VOLCANO_POINTS_ONLY = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_volcano_points_only.png"


# ============================================
# HELPERS
# ============================================
def first_gene_symbol(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    s = s.split(";")[0]
    s = s.split(" ")[0]
    return s.upper()

def p_to_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

# ============================================
# FIND SHEETS
# ============================================
xls = pd.ExcelFile(DATA_FILE)
sheet_names = xls.sheet_names
print("Workbook sheets:", sheet_names)

sheet_lpspbs = None
sheet_lpsvpbs = None

for s in sheet_names:
    s_low = s.lower()
    if "lps+pbs" in s_low:
        sheet_lpspbs = s
    if "lpsvpbs" in s_low and "ipa" not in s_low:
        sheet_lpsvpbs = s

if sheet_lpspbs is None:
    raise ValueError("Could not find the LPS+PBS sheet.")
if sheet_lpsvpbs is None:
    raise ValueError("Could not find the Endfoot_LPSvPBS sheet.")

print("Using filter sheet:", sheet_lpspbs)
print("Using comparison sheet:", sheet_lpsvpbs)

# ============================================
# LOAD FILTER SHEET (LPS+PBS)
# Keep only TurboID enriched or TurboID unique
# ============================================
df_filter = pd.read_excel(DATA_FILE, sheet_name=sheet_lpspbs, header=3)
df_filter.columns = df_filter.columns.str.strip()

required_filter_cols = ["Mouse.Gene", "Differentially Expressed?"]
for col in required_filter_cols:
    if col not in df_filter.columns:
        raise ValueError(f"Missing expected column in filter sheet: {col}")

df_filter["Gene_clean"] = df_filter["Mouse.Gene"].apply(first_gene_symbol)

keep_classes = ["TurboID enriched", "TurboID unique"]
df_filter_kept = df_filter[df_filter["Differentially Expressed?"].isin(keep_classes)].copy()

turboid_genes = set(df_filter_kept["Gene_clean"].dropna())
print("TurboID-enriched/unique genes:", len(turboid_genes))

# ============================================
# LOAD COMPARISON SHEET (Endfoot_LPSvPBS)
# Use pipeline averages and stats directly
# ============================================
df_cmp = pd.read_excel(DATA_FILE, sheet_name=sheet_lpsvpbs, header=2)
df_cmp.columns = df_cmp.columns.str.strip()

required_cmp_cols = [
    "Mouse.Gene",
    "LPS avg",
    "PBS avg",
    "logFC (LPS/PBS)",
    "P.Value",
    "adj.P.Value"
]
for col in required_cmp_cols:
    if col not in df_cmp.columns:
        raise ValueError(f"Missing expected column in comparison sheet: {col}")

df_cmp["Gene_clean"] = df_cmp["Mouse.Gene"].apply(first_gene_symbol)

# Keep only TurboID-selected proteins
df_cmp = df_cmp[df_cmp["Gene_clean"].isin(turboid_genes)].copy()
print("After TurboID filter:", df_cmp.shape)

# Convert relevant columns to numeric
numeric_cols = ["LPS avg", "PBS avg", "logFC (LPS/PBS)", "P.Value", "adj.P.Value"]
for c in numeric_cols:
    df_cmp[c] = pd.to_numeric(df_cmp[c], errors="coerce")

# ============================================
# LOAD STAT3 LIST
# ============================================
stat3 = pd.read_excel(STAT3_FILE)
stat3.columns = stat3.columns.str.strip()

possible_gene_cols = [
    "Gene",
    "Gene_symbol",
    "Gene Symbol",
    "Genes",
    "Mouse_gene_symbol",
    "Receptor_gene_symbol_mouse"
]

stat3_gene_col = None
for c in possible_gene_cols:
    if c in stat3.columns:
        stat3_gene_col = c
        break

if stat3_gene_col is None:
    stat3_gene_col = stat3.columns[0]

print("Using STAT3 gene column:", stat3_gene_col)

stat3_genes = set(
    stat3[stat3_gene_col]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

# ============================================
# CROSS-REFERENCE WITH STAT3 LIST
# ============================================
df_stat3 = df_cmp[df_cmp["Gene_clean"].isin(stat3_genes)].copy()
print("STAT3-relevant TurboID proteins:", df_stat3.shape)

if df_stat3.shape[0] == 0:
    raise ValueError("No STAT3-relevant proteins found after filtering.")

# ============================================
# PREPARE FINAL TABLE
# ============================================
df_stat3["stars"] = df_stat3["P.Value"].apply(p_to_stars)
df_stat3["neglog10_pvalue"] = -np.log10(df_stat3["P.Value"].replace(0, np.nan))

# Optional display values for the bar plot
df_stat3["PBS avg log2"] = np.log2(df_stat3["PBS avg"] + 1)
df_stat3["LPS avg log2"] = np.log2(df_stat3["LPS avg"] + 1)

# Keep only useful columns in the exported table
keep_cols = [
    "Gene_clean",
    "Mouse.Gene",
    "Human.Gene",
    "Uniprot ID",
    "Protein Name",
    "Protein description",
    "PBS avg",
    "LPS avg",
    "PBS avg log2",
    "LPS avg log2",
    "logFC (LPS/PBS)",
    "P.Value",
    "adj.P.Value",
    "stars"
]
keep_cols = [c for c in keep_cols if c in df_stat3.columns]

df_stat3 = df_stat3[keep_cols + ["neglog10_pvalue"]].copy()
df_stat3 = df_stat3.sort_values(["P.Value", "logFC (LPS/PBS)"], ascending=[True, False])

# Save filtered table
df_stat3.to_excel(OUT_TABLE, index=False)

# ============================================
# VOLCANO PLOT PNG
# ============================================
plt.figure(figsize=(8, 6))
plt.scatter(df_stat3["logFC (LPS/PBS)"], df_stat3["neglog10_pvalue"], s=45)

for _, row in df_stat3.iterrows():
    label = str(row["Gene_clean"])
    if row["stars"] != "":
        label = f"{label} {row['stars']}"
    plt.text(
        row["logFC (LPS/PBS)"],
        row["neglog10_pvalue"],
        label,
        fontsize=8
    )

plt.axvline(1, linestyle="--", linewidth=1)
plt.axvline(-1, linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), linestyle="--", linewidth=1)

plt.xlabel("logFC (LPS/PBS)")
plt.ylabel("-log10 P.Value")
plt.title("STAT3-relevant TurboID endfoot proteins")
plt.tight_layout()
plt.savefig(OUT_VOLCANO, dpi=300, bbox_inches="tight")
plt.close()


# ============================================
# VOLCANO PLOT (POINTS ONLY - NO LABELS)
# ============================================
plt.figure(figsize=(8, 6))

plt.scatter(
    df_stat3["logFC (LPS/PBS)"],
    df_stat3["neglog10_pvalue"],
    s=45
)

# thresholds
plt.axvline(1, linestyle="--", linewidth=1)
plt.axvline(-1, linestyle="--", linewidth=1)
plt.axhline(-np.log10(0.05), linestyle="--", linewidth=1)

plt.xlabel("logFC (LPS/PBS)")
plt.ylabel("-log10 P.Value")
plt.title("STAT3-relevant TurboID endfoot proteins (points only)")

plt.tight_layout()
plt.savefig(OUT_VOLCANO_POINTS_ONLY, dpi=300, bbox_inches="tight")
plt.close()


# ============================================
# BAR PLOT PNG
# ============================================
bar_df = df_stat3[[
    "Gene_clean",
    "PBS avg log2",
    "LPS avg log2",
    "stars"
]].copy()

bar_df = bar_df.sort_values("LPS avg log2", ascending=False).reset_index(drop=True)

x = np.arange(len(bar_df))
width = 0.38

plt.figure(figsize=(max(10, len(bar_df) * 0.45), 6))
plt.bar(x - width/2, bar_df["PBS avg log2"], width=width, label="PBS")
plt.bar(x + width/2, bar_df["LPS avg log2"], width=width, label="LPS")

for i, row in bar_df.iterrows():
    ymax = max(row["PBS avg log2"], row["LPS avg log2"])
    if row["stars"] != "":
        plt.text(i, ymax + 0.12, row["stars"], ha="center", va="bottom", fontsize=10)

plt.xticks(x, bar_df["Gene_clean"], rotation=90)
plt.ylabel("Mean log2 protein expression")
plt.title("STAT3-relevant TurboID endfoot proteins: PBS vs LPS")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_BARPLOT, dpi=300, bbox_inches="tight")
plt.close()

# ============================================
# EDITABLE PPTX CHARTS
# ============================================
prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

# ---- Slide 1: Editable volcano scatter ----
slide = prs.slides.add_slide(prs.slide_layouts[6])

title_box = slide.shapes.add_textbox(Inches(0.4), Inches(0.2), Inches(12.0), Inches(0.5))
title_box.text_frame.text = "STAT3-relevant TurboID endfoot proteins - Volcano plot"

xy_data = XyChartData()
series = xy_data.add_series("STAT3 proteins")

for _, row in df_stat3.iterrows():
    xval = float(row["logFC (LPS/PBS)"]) if pd.notna(row["logFC (LPS/PBS)"]) else 0.0
    yval = float(row["neglog10_pvalue"]) if pd.notna(row["neglog10_pvalue"]) else 0.0
    series.add_data_point(xval, yval)

chart = slide.shapes.add_chart(
    XL_CHART_TYPE.XY_SCATTER,
    Inches(0.7), Inches(0.9), Inches(8.7), Inches(5.8),
    xy_data
).chart

chart.has_legend = False
chart.chart_title.has_text_frame = True
chart.chart_title.text_frame.text = "logFC vs -log10(P.Value)"

value_axis = chart.value_axis
value_axis.has_title = True
value_axis.axis_title.text_frame.text = "-log10 P.Value"

category_axis = chart.category_axis
category_axis.has_title = True
category_axis.axis_title.text_frame.text = "logFC (LPS/PBS)"

# add gene labels as editable text boxes next to points
# simple coordinate transform from data space to slide space
x_min = min(-1.5, float(df_stat3["logFC (LPS/PBS)"].min()) - 0.3)
x_max = max(1.5, float(df_stat3["logFC (LPS/PBS)"].max()) + 0.3)
y_min = 0.0
y_max = max(1.5, float(df_stat3["neglog10_pvalue"].max()) + 0.5)

plot_left = 0.7
plot_top = 0.9
plot_w = 8.7
plot_h = 5.8

for _, row in df_stat3.iterrows():
    xval = float(row["logFC (LPS/PBS)"]) if pd.notna(row["logFC (LPS/PBS)"]) else 0.0
    yval = float(row["neglog10_pvalue"]) if pd.notna(row["neglog10_pvalue"]) else 0.0
    label = str(row["Gene_clean"])
    if row["stars"] != "":
        label = f"{label} {row['stars']}"

    # normalize into chart box
    x_norm = (xval - x_min) / (x_max - x_min) if x_max > x_min else 0.5
    y_norm = (yval - y_min) / (y_max - y_min) if y_max > y_min else 0.5

    # powerpoint origin is top-left, y inverted
    left = plot_left + x_norm * plot_w
    top = plot_top + (1 - y_norm) * plot_h

    tb = slide.shapes.add_textbox(Inches(left + 0.02), Inches(top - 0.02), Inches(1.0), Inches(0.2))
    p = tb.text_frame.paragraphs[0]
    p.text = label
    p.font.size = Pt(8)

# ---- Slide 2: Editable bar chart ----
slide = prs.slides.add_slide(prs.slide_layouts[6])

title_box = slide.shapes.add_textbox(Inches(0.4), Inches(0.2), Inches(12.0), Inches(0.5))
title_box.text_frame.text = "STAT3-relevant TurboID endfoot proteins - Bar plot"

chart_data = CategoryChartData()
chart_data.categories = list(bar_df["Gene_clean"])
chart_data.add_series("PBS", list(bar_df["PBS avg log2"]))
chart_data.add_series("LPS", list(bar_df["LPS avg log2"]))

chart = slide.shapes.add_chart(
    XL_CHART_TYPE.COLUMN_CLUSTERED,
    Inches(0.6), Inches(0.9), Inches(12.0), Inches(5.8),
    chart_data
).chart

chart.has_legend = True
chart.legend.position = XL_LEGEND_POSITION.BOTTOM
chart.legend.include_in_layout = False

chart.value_axis.has_title = True
chart.value_axis.axis_title.text_frame.text = "Mean log2 protein expression"

# add editable significance stars above bars as text boxes
n = len(bar_df)
for i, row in bar_df.iterrows():
    if row["stars"] == "":
        continue

    # approximate placement across chart width
    frac = (i + 0.5) / max(n, 1)
    left = 0.6 + frac * 12.0 - 0.12
    top = 1.0

    tb = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(0.3), Inches(0.2))
    p = tb.text_frame.paragraphs[0]
    p.text = row["stars"]
    p.font.size = Pt(10)

prs.save(OUT_PPTX)

# ============================================
# SUMMARY
# ============================================
print("\nSaved:")
print(OUT_TABLE)
print(OUT_VOLCANO)
print(OUT_BARPLOT)
print(OUT_PPTX)
print(OUT_VOLCANO_POINTS_ONLY)

print("\nPreview:")
print(df_stat3[[
    "Gene_clean",
    "PBS avg",
    "LPS avg",
    "logFC (LPS/PBS)",
    "P.Value",
    "adj.P.Value",
    "stars"
]].head(20))

Workbook sheets: ['readme', 'Endfoot_LPSvPBS (2756)', 'IPA_sig_Endfoot_LPSvPBS (2756)', 'Endfoot LPS+PBS (3849)']
Using filter sheet: Endfoot LPS+PBS (3849)
Using comparison sheet: Endfoot_LPSvPBS (2756)
TurboID-enriched/unique genes: 3142
After TurboID filter: (2290, 23)
Using STAT3 gene column: Gene
STAT3-relevant TurboID proteins: (20, 23)

Saved:
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_LPSvPBS_filtered.xlsx
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_volcano_pipeline_stats.png
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_barplot_pipeline_stats.png
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_editable_charts.pptx
/Users/antina/Desktop/NeuroPSI/data mining/STAT3_TurboID_endfoot_volcano_points_only.png

Preview:
     Gene_clean       PBS avg       LPS avg  logFC (LPS/PBS)   P.Value  \
2350      STAT3   4177.965989  15330.866261         1.901134  0.006433   
1841      PTPRF   8766.530459  15302.606338

In [ ]:
# ============================================
# 4) Astro LPS VS PBS (From Sample mean)
# ============================================

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import openpyxl
import matplotlib.pyplot as plt


# ---------------------------
# User-configurable file paths
# ---------------------------
MATRIX_FILE = Path("/Users/antina/Desktop/NeuroPSI/data mining/proteomics_matrix_with_descriptive_sample_names.xlsx")
ASTRO_STATS_FILE = Path("/Users/antina/Desktop/NeuroPSI/data mining/Asrocytes-proteom-stats.xlsx")
STAT3_FILE = Path("/Users/antina/Desktop/NeuroPSI/data mining/STAT3_full_pathway_plus_receptors_mouse.xlsx")

OUTPUT_EXCEL = Path("/Users/antina/Desktop/NeuroPSI/data mining/astro_LPS_vs_PBS_STAT3_hits_only.xlsx")
OUTPUT_PLOT = Path("/Users/antina/Desktop/NeuroPSI/data mining/astro_LPS_vs_PBS_STAT3_barplot.png")

# ---------------------------
# Filtering parameters
# ---------------------------
# Keep proteins only if they have >= 3 quantified values overall across selected astro LPS/PBS samples
MIN_NONMISSING_TOTAL = 3

# For group statistics / Welch t-test, require at least 2 quantified values in each group
MIN_NONMISSING_PER_GROUP = 2

# Pseudocount used only when converting to log2 values if needed
LOG2_PSEUDOCOUNT = 1e-9

# Significance threshold for plot annotation
PVALUE_THRESHOLD = 0.05


def normalize_gene_token(value):
    """Normalize a gene/protein token for matching."""
    if pd.isna(value):
        return None
    value = str(value).strip()
    if not value:
        return None
    return value.lower()


def split_tokens(value):
    """Split semicolon/comma/pipe-separated entries into normalized tokens."""
    if pd.isna(value):
        return []
    tokens = re.split(r"[;,|]", str(value))
    cleaned = []
    for token in tokens:
        norm = normalize_gene_token(token)
        if norm:
            cleaned.append(norm)
    return cleaned


def find_astro_columns_excluding_yellow(xlsx_path):
    """
    Read workbook formatting and return astrocyte sample columns except yellow-highlighted ones.
    Yellow highlighted columns are assumed to be TdTomato controls to exclude.
    """
    wb = openpyxl.load_workbook(xlsx_path)
    ws = wb[wb.sheetnames[0]]

    selected = []
    excluded_yellow = []

    for col_idx in range(1, ws.max_column + 1):
        header = ws.cell(1, col_idx).value
        if not isinstance(header, str):
            continue

        if not header.startswith("astrocyte-"):
            continue

        fill = ws.cell(1, col_idx).fill
        fg = getattr(fill.fgColor, "rgb", None)
        is_yellow = (fill.patternType == "solid" and fg is not None and fg.upper() == "FFFFFF00")

        if is_yellow:
            excluded_yellow.append(header)
        else:
            selected.append(header)

    return selected, excluded_yellow


def load_tdTomato_exclusions(stats_path):
    """
    Load genes/proteins flagged as tdTomato enriched/unique from the astro proteome stats sheet.
    """
    stats_df = pd.read_excel(stats_path, sheet_name="Astro proteins (5061 proteins)", header=2)

    td_mask = stats_df["Differentially Expressed?"].isin(["tdTomato enriched", "tdTomato unique"])
    td_df = stats_df.loc[td_mask].copy()

    td_genes = set(
        normalize_gene_token(g)
        for g in td_df["Mouse.Gene"].dropna().tolist()
        if normalize_gene_token(g) is not None
    )
    td_uniprot = set(
        normalize_gene_token(u)
        for u in td_df["Uniprot ID"].dropna().tolist()
        if normalize_gene_token(u) is not None
    )

    return td_df, td_genes, td_uniprot


def add_significance_annotations(ax, x1, x2, y, h, text):
    """Draw significance bracket and text above two bars."""
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], linewidth=1)
    ax.text((x1 + x2) / 2, y + h, text, ha='center', va='bottom', fontsize=8)


def make_barplot(stat3_hits_df, output_path):
    """
    Create a grouped bar plot of mean PBS vs mean LPS expression
    for STAT3-matched proteins, and annotate significant p-values.
    """
    if stat3_hits_df.empty:
        print("No STAT3 hits found; skipping plot.")
        return

    plot_df = stat3_hits_df.copy()

    # Create clean labels
    plot_df["Plot_Label"] = plot_df["STAT3_Genes_Matched"].replace("", np.nan)
    plot_df["Plot_Label"] = plot_df["Plot_Label"].fillna(plot_df["Genes"])
    plot_df["Plot_Label"] = plot_df["Plot_Label"].fillna(plot_df["Protein_label"])

    # If duplicates exist, make labels unique
    counts = {}
    unique_labels = []
    for lbl in plot_df["Plot_Label"]:
        if lbl in counts:
            counts[lbl] += 1
            unique_labels.append(f"{lbl}_{counts[lbl]}")
        else:
            counts[lbl] = 1
            unique_labels.append(lbl)
    plot_df["Plot_Label"] = unique_labels

    # Sort by p-value then FC for readability
    plot_df = plot_df.sort_values(
        by=["P_value", "log2_FC_LPS_vs_PBS"],
        ascending=[True, False],
        na_position="last"
    ).reset_index(drop=True)

    x = np.arange(len(plot_df))
    width = 0.38

    fig_width = max(10, len(plot_df) * 0.7)
    fig, ax = plt.subplots(figsize=(fig_width, 6))

    ax.bar(x - width / 2, plot_df["Mean_PBS"], width, label="PBS")
    ax.bar(x + width / 2, plot_df["Mean_LPS"], width, label="LPS")

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["Plot_Label"], rotation=45, ha="right")
    ax.set_ylabel("Mean expression")
    ax.set_title("STAT3-matched proteins: astrocyte LPS vs PBS")
    ax.legend()

    # Annotate significant p-values
    y_max = max(plot_df["Mean_PBS"].max(), plot_df["Mean_LPS"].max())
    base_offset = y_max * 0.03 if y_max > 0 else 0.1

    for i, row in plot_df.iterrows():
        pval = row["P_value"]
        if pd.notna(pval) and pval < PVALUE_THRESHOLD:
            top = max(row["Mean_PBS"], row["Mean_LPS"])
            text = f"p={pval:.3g}"
            add_significance_annotations(
                ax,
                i - width / 2,
                i + width / 2,
                y=top + base_offset,
                h=base_offset * 0.7,
                text=text
            )

    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Bar plot saved to: {output_path}")


def main():
    # 1) Determine usable astrocyte columns from formatting
    astro_cols, yellow_cols = find_astro_columns_excluding_yellow(MATRIX_FILE)

    pbs_cols = [c for c in astro_cols if "-PBS-" in c]
    lps_cols = [c for c in astro_cols if "-LPS-" in c]

    if len(pbs_cols) == 0 or len(lps_cols) == 0:
        raise ValueError("Could not identify both PBS and LPS astrocyte sample columns.")

    # 2) Load main matrix
    df = pd.read_excel(MATRIX_FILE)

    required_id_cols = [
        "Protein_label",
        "Protein.Group",
        "Protein.Ids",
        "Protein.Names",
        "Genes",
        "First.Protein.Description",
    ]
    missing_id_cols = [c for c in required_id_cols if c not in df.columns]
    if missing_id_cols:
        raise ValueError(f"Missing expected columns in matrix file: {missing_id_cols}")

    # Keep only identifiers + selected astro sample columns
    working = df[required_id_cols + astro_cols].copy()

    # 3) Remove proteins with too few quantified values
    working["n_nonmissing_total"] = working[astro_cols].notna().sum(axis=1)
    working["n_nonmissing_pbs"] = working[pbs_cols].notna().sum(axis=1)
    working["n_nonmissing_lps"] = working[lps_cols].notna().sum(axis=1)

    keep_mask = (
        (working["n_nonmissing_total"] >= MIN_NONMISSING_TOTAL)
        & (working["n_nonmissing_pbs"] >= MIN_NONMISSING_PER_GROUP)
        & (working["n_nonmissing_lps"] >= MIN_NONMISSING_PER_GROUP)
    )
    filtered = working.loc[keep_mask].copy()

    # 4) Remove proteins that are tdTomato-enriched or tdTomato-unique
    td_df, td_genes, td_uniprot = load_tdTomato_exclusions(ASTRO_STATS_FILE)

    def is_tdTomato_flagged(row):
        gene_tokens = split_tokens(row["Genes"])
        uniprot_tokens = split_tokens(row["Protein.Ids"]) + split_tokens(row["Protein.Group"])
        gene_hit = any(tok in td_genes for tok in gene_tokens)
        uniprot_hit = any(tok in td_uniprot for tok in uniprot_tokens)
        return gene_hit or uniprot_hit

    filtered["Excluded_tdTomato"] = filtered.apply(is_tdTomato_flagged, axis=1)
    filtered = filtered.loc[~filtered["Excluded_tdTomato"]].copy()

    # 5) Compute means, log2 means, log2 FC, and p-values
    filtered["Mean_PBS"] = filtered[pbs_cols].mean(axis=1, skipna=True)
    filtered["Mean_LPS"] = filtered[lps_cols].mean(axis=1, skipna=True)

    log2_pbs = np.log2(filtered[pbs_cols].astype(float) + LOG2_PSEUDOCOUNT)
    log2_lps = np.log2(filtered[lps_cols].astype(float) + LOG2_PSEUDOCOUNT)

    filtered["Log2Mean_PBS"] = log2_pbs.mean(axis=1, skipna=True)
    filtered["Log2Mean_LPS"] = log2_lps.mean(axis=1, skipna=True)
    filtered["log2_FC_LPS_vs_PBS"] = filtered["Log2Mean_LPS"] - filtered["Log2Mean_PBS"]

    p_values = []
    for idx in filtered.index:
        pbs_vals = log2_pbs.loc[idx].dropna().values
        lps_vals = log2_lps.loc[idx].dropna().values

        if len(pbs_vals) >= MIN_NONMISSING_PER_GROUP and len(lps_vals) >= MIN_NONMISSING_PER_GROUP:
            _, pval = ttest_ind(lps_vals, pbs_vals, equal_var=False, nan_policy="omit")
        else:
            pval = np.nan
        p_values.append(pval)

    filtered["P_value"] = p_values

    # 6) Cross-link with STAT3 list
    stat3_df = pd.read_excel(STAT3_FILE).copy()
    stat3_genes = set(normalize_gene_token(g) for g in stat3_df["Gene"].dropna())

    def extract_matching_stat3_genes(gene_value):
        matches = []
        for tok in split_tokens(gene_value):
            if tok in stat3_genes:
                matches.append(tok)
        # preserve order but deduplicate
        seen = set()
        ordered = []
        for m in matches:
            if m not in seen:
                seen.add(m)
                ordered.append(m)
        return ordered

    filtered["STAT3_match_tokens"] = filtered["Genes"].apply(extract_matching_stat3_genes)
    filtered["STAT3_match"] = filtered["STAT3_match_tokens"].apply(lambda x: len(x) > 0)
    filtered["STAT3_Genes_Matched"] = filtered["STAT3_match_tokens"].apply(
        lambda x: "; ".join(x) if x else ""
    )

    stat3_annot = stat3_df.copy()
    stat3_annot["Gene_norm"] = stat3_annot["Gene"].apply(normalize_gene_token)

    def get_stat3_categories(matches):
        if not matches:
            return ""
        vals = stat3_annot.loc[
            stat3_annot["Gene_norm"].isin(matches), "Category"
        ].dropna().astype(str).tolist()
        return "; ".join(dict.fromkeys(vals))

    def get_stat3_roles(matches):
        if not matches:
            return ""
        vals = stat3_annot.loc[
            stat3_annot["Gene_norm"].isin(matches), "Role_in_JAK_STAT"
        ].dropna().astype(str).tolist()
        return "; ".join(dict.fromkeys(vals))

    filtered["STAT3_Category"] = filtered["STAT3_match_tokens"].apply(get_stat3_categories)
    filtered["STAT3_Role"] = filtered["STAT3_match_tokens"].apply(get_stat3_roles)

    # 7) Keep only STAT3 hits
    final_cols = required_id_cols + [
        "n_nonmissing_total",
        "n_nonmissing_pbs",
        "n_nonmissing_lps",
        "Mean_PBS",
        "Mean_LPS",
        "Log2Mean_PBS",
        "Log2Mean_LPS",
        "log2_FC_LPS_vs_PBS",
        "P_value",
        "STAT3_Genes_Matched",
        "STAT3_Category",
        "STAT3_Role",
    ] + pbs_cols + lps_cols

    stat3_hits_df = filtered.loc[filtered["STAT3_match"], final_cols].copy()
    stat3_hits_df = stat3_hits_df.sort_values(
        by=["P_value", "log2_FC_LPS_vs_PBS"],
        ascending=[True, False],
        na_position="last",
    )

    # 8) Export ONLY the STAT3 hits
    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
        stat3_hits_df.to_excel(writer, sheet_name="STAT3_hits_only", index=False)

    print(f"Done. Wrote STAT3-only results to: {OUTPUT_EXCEL}")

    # 9) Bar plot
    make_barplot(stat3_hits_df, OUTPUT_PLOT)


if __name__ == "__main__":
    main()


In [ ]:
# ============================================
# 5) Ebrichment Astro VS endfeet LPS and PBS (From idividual sample value)
# ============================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
import re
from pptx import Presentation
from pptx.util import Inches
from scipy.stats import t
from pptx.dml.color import RGBColor
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE

# =========================
# FILE PATHS
# =========================
PROTEOMICS_FILE = "/Users/antina/Desktop/NeuroPSI/data mining/proteomics_matrix_with_descriptive_sample_names.xlsx"
STAT3_FILE = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_full_pathway_plus_receptors_mouse.xlsx"

OUT_TABLE = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_endfoot_vs_astrocyte_enrichment.xlsx"
OUT_BARPLOT_PBS = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_endfoot_vs_astrocyte_enrichment_PBS.png"
OUT_BARPLOT_LPS = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_endfoot_vs_astrocyte_enrichment_LPS.png"
OUT_BARPLOT_ALL = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_endfoot_vs_astrocyte_enrichment_all.png"
OUT_HEATMAP = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_endfoot_vs_astrocyte_expression_heatmap.png"
OUT_PPTX = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_endfoot_vs_astrocyte_charts.pptx"
OUT_HEATMAP_STATS = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_heatmap_statistics.xlsx"
OUT_BARPLOT_PBS_LPS_MERGED = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_endfoot_vs_astrocyte_enrichment_PBS_LPS_merged.png"
OUT_BARPLOT_STAT3_ONLY = "/Users/antina/Desktop/NeuroPSI/data mining/STAT3_only_PvAP_vs_astrocyte_enrichment_VEH_LPS.png"

# =========================
# tdTOMATO CONTROL SAMPLE IDS
# =========================
tdtomato_ids = {"149", "150", "152", "153", "154", "161", "162"}

def is_tdtomato_col(col):
    col = str(col)
    numbers = re.findall(r"\d+", col)
    return any(num in tdtomato_ids for num in numbers)

def is_analysis_col(col):
    return not is_tdtomato_col(col)

# =========================
# FDR FUNCTION
# =========================
def benjamini_hochberg(pvalues):
    pvalues = np.array(pvalues, dtype=float)
    n = len(pvalues)

    order = np.argsort(pvalues)
    ranked = pvalues[order]

    qvals = np.empty(n, dtype=float)
    prev_q = 1.0

    for i in range(n - 1, -1, -1):
        rank = i + 1
        q = ranked[i] * n / rank
        q = min(q, prev_q)
        prev_q = q
        qvals[i] = q

    out = np.empty(n, dtype=float)
    out[order] = qvals
    return out

# =========================
# LOAD DATA
# =========================
prot = pd.read_excel(PROTEOMICS_FILE)
stat3 = pd.read_excel(STAT3_FILE)

print("Proteomics shape:", prot.shape)
print("STAT3 list shape:", stat3.shape)

# =========================
# FIND STAT3 GENE COLUMN
# =========================
possible_gene_cols = [
    "Gene",
    "Gene_symbol",
    "Gene Symbol",
    "gene",
    "Genes",
    "Mouse_gene_symbol",
    "Receptor_gene_symbol_mouse"
]

stat3_gene_col = None
for c in possible_gene_cols:
    if c in stat3.columns:
        stat3_gene_col = c
        break

if stat3_gene_col is None:
    stat3_gene_col = stat3.columns[0]

print("Using STAT3 gene column:", stat3_gene_col)

stat3_genes = (
    stat3[stat3_gene_col]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
)

# =========================
# IDENTIFY SAMPLE COLUMNS
# =========================
astro_lps_cols = [
    c for c in prot.columns
    if isinstance(c, str)
    and c.startswith("astrocyte-LPS")
    and is_analysis_col(c)
]

astro_pbs_cols = [
    c for c in prot.columns
    if isinstance(c, str)
    and c.startswith("astrocyte-PBS")
    and is_analysis_col(c)
]

endfoot_lps_cols = [
    c for c in prot.columns
    if isinstance(c, str)
    and c.startswith("endfoot-LPS")
    and is_analysis_col(c)
]

endfoot_pbs_cols = [
    c for c in prot.columns
    if isinstance(c, str)
    and c.startswith("endfoot-PBS")
    and is_analysis_col(c)
]

td_astro_lps_cols = [
    c for c in prot.columns
    if isinstance(c, str)
    and c.startswith("astrocyte-LPS")
    and is_tdtomato_col(c)
]

td_astro_pbs_cols = [
    c for c in prot.columns
    if isinstance(c, str)
    and c.startswith("astrocyte-PBS")
    and is_tdtomato_col(c)
]

td_endfoot_lps_cols = [
    c for c in prot.columns
    if isinstance(c, str)
    and c.startswith("endfoot-LPS")
    and is_tdtomato_col(c)
]

td_endfoot_pbs_cols = [
    c for c in prot.columns
    if isinstance(c, str)
    and c.startswith("endfoot-PBS")
    and is_tdtomato_col(c)
]

astro_cols = astro_lps_cols + astro_pbs_cols
endfoot_cols = endfoot_lps_cols + endfoot_pbs_cols
all_sample_cols = astro_cols + endfoot_cols

td_cols = (
    td_astro_lps_cols
    + td_astro_pbs_cols
    + td_endfoot_lps_cols
    + td_endfoot_pbs_cols
)

all_numeric_cols = all_sample_cols + td_cols

print("Astrocyte PBS used:", astro_pbs_cols)
print("Astrocyte LPS used:", astro_lps_cols)
print("Endfoot PBS used:", endfoot_pbs_cols)
print("Endfoot LPS used:", endfoot_lps_cols)

print("tdTomato astrocyte PBS controls:", td_astro_pbs_cols)
print("tdTomato astrocyte LPS controls:", td_astro_lps_cols)
print("tdTomato endfoot PBS controls:", td_endfoot_pbs_cols)
print("tdTomato endfoot LPS controls:", td_endfoot_lps_cols)

if len(astro_cols) == 0 or len(endfoot_cols) == 0:
    raise ValueError("Could not find astrocyte and/or endfoot analysis sample columns.")

if (
    len(td_astro_pbs_cols) == 0
    or len(td_astro_lps_cols) == 0
    or len(td_endfoot_pbs_cols) == 0
    or len(td_endfoot_lps_cols) == 0
):
    raise ValueError(
        "Some tdTomato control columns were not detected. "
        "Check tdtomato_ids and column names."
    )

# =========================
# PREPARE GENE COLUMN
# =========================
if "Genes" not in prot.columns:
    raise ValueError("The proteomics file must contain a 'Genes' column.")

prot = prot.copy()
prot["Genes_clean"] = prot["Genes"].astype(str).str.strip().str.upper()
prot["Genes_first"] = prot["Genes_clean"].str.split(";").str[0].str.split(" ").str[0]

for c in all_numeric_cols:
    prot[c] = pd.to_numeric(prot[c], errors="coerce")

# =========================
# KEEP ONLY STAT3-RELEVANT PROTEINS
# =========================
stat3_prot = prot[prot["Genes_first"].isin(stat3_genes)].copy()
stat3_prot = stat3_prot.dropna(subset=all_sample_cols, how="all").copy()

if stat3_prot.shape[0] == 0:
    raise ValueError("No STAT3-relevant proteins found in the proteomics file.")

print("STAT3 proteins found before tdTomato filtering:", stat3_prot.shape[0])

# =========================
# LOG2 TRANSFORM
# =========================
for c in all_numeric_cols:
    stat3_prot[c] = np.log2(stat3_prot[c] + 1)

# =========================
# COMPUTE MEAN EXPRESSION
# =========================
stat3_prot["mean_astro_PBS"] = stat3_prot[astro_pbs_cols].mean(axis=1)
stat3_prot["mean_astro_LPS"] = stat3_prot[astro_lps_cols].mean(axis=1)
stat3_prot["mean_astro_all"] = stat3_prot[astro_cols].mean(axis=1)

stat3_prot["mean_endfoot_PBS"] = stat3_prot[endfoot_pbs_cols].mean(axis=1)
stat3_prot["mean_endfoot_LPS"] = stat3_prot[endfoot_lps_cols].mean(axis=1)
stat3_prot["mean_endfoot_all"] = stat3_prot[endfoot_cols].mean(axis=1)

# =========================
# tdTOMATO BACKGROUND FILTER
# =========================
MIN_TD_REPLICATES = 2

stat3_prot["mean_astro_PBS_raw"] = stat3_prot["mean_astro_PBS"]
stat3_prot["mean_astro_LPS_raw"] = stat3_prot["mean_astro_LPS"]
stat3_prot["mean_endfoot_PBS_raw"] = stat3_prot["mean_endfoot_PBS"]
stat3_prot["mean_endfoot_LPS_raw"] = stat3_prot["mean_endfoot_LPS"]

stat3_prot["mean_td_astro_PBS"] = stat3_prot[td_astro_pbs_cols].mean(axis=1)
stat3_prot["mean_td_astro_LPS"] = stat3_prot[td_astro_lps_cols].mean(axis=1)
stat3_prot["mean_td_endfoot_PBS"] = stat3_prot[td_endfoot_pbs_cols].mean(axis=1)
stat3_prot["mean_td_endfoot_LPS"] = stat3_prot[td_endfoot_lps_cols].mean(axis=1)

stat3_prot["n_td_astro_PBS"] = stat3_prot[td_astro_pbs_cols].notna().sum(axis=1)
stat3_prot["n_td_astro_LPS"] = stat3_prot[td_astro_lps_cols].notna().sum(axis=1)
stat3_prot["n_td_endfoot_PBS"] = stat3_prot[td_endfoot_pbs_cols].notna().sum(axis=1)
stat3_prot["n_td_endfoot_LPS"] = stat3_prot[td_endfoot_lps_cols].notna().sum(axis=1)

stat3_prot["pass_td_astro_PBS"] = np.where(
    stat3_prot["n_td_astro_PBS"] >= MIN_TD_REPLICATES,
    stat3_prot["mean_astro_PBS_raw"] > stat3_prot["mean_td_astro_PBS"],
    True
)

stat3_prot["pass_td_astro_LPS"] = np.where(
    stat3_prot["n_td_astro_LPS"] >= MIN_TD_REPLICATES,
    stat3_prot["mean_astro_LPS_raw"] > stat3_prot["mean_td_astro_LPS"],
    True
)

stat3_prot["pass_td_endfoot_PBS"] = np.where(
    stat3_prot["n_td_endfoot_PBS"] >= MIN_TD_REPLICATES,
    stat3_prot["mean_endfoot_PBS_raw"] > stat3_prot["mean_td_endfoot_PBS"],
    True
)

stat3_prot["pass_td_endfoot_LPS"] = np.where(
    stat3_prot["n_td_endfoot_LPS"] >= MIN_TD_REPLICATES,
    stat3_prot["mean_endfoot_LPS_raw"] > stat3_prot["mean_td_endfoot_LPS"],
    True
)

stat3_prot.loc[~stat3_prot["pass_td_astro_PBS"], "mean_astro_PBS"] = np.nan
stat3_prot.loc[~stat3_prot["pass_td_astro_LPS"], "mean_astro_LPS"] = np.nan
stat3_prot.loc[~stat3_prot["pass_td_endfoot_PBS"], "mean_endfoot_PBS"] = np.nan
stat3_prot.loc[~stat3_prot["pass_td_endfoot_LPS"], "mean_endfoot_LPS"] = np.nan

stat3_prot["mean_astro_all"] = stat3_prot[
    ["mean_astro_PBS", "mean_astro_LPS"]
].mean(axis=1)

stat3_prot["mean_endfoot_all"] = stat3_prot[
    ["mean_endfoot_PBS", "mean_endfoot_LPS"]
].mean(axis=1)

print("Condition-specific tdTomato filtering applied.")
print("Minimum tdTomato replicates required:", MIN_TD_REPLICATES)
print("Proteins retained in astro PBS:", stat3_prot["mean_astro_PBS"].notna().sum())
print("Proteins retained in astro LPS:", stat3_prot["mean_astro_LPS"].notna().sum())
print("Proteins retained in endfoot PBS:", stat3_prot["mean_endfoot_PBS"].notna().sum())
print("Proteins retained in endfoot LPS:", stat3_prot["mean_endfoot_LPS"].notna().sum())

# =========================
# ENRICHMENT SCORES
# =========================
stat3_prot["enrichment_PBS_endfoot_vs_astro"] = (
    stat3_prot["mean_endfoot_PBS"] - stat3_prot["mean_astro_PBS"]
)

stat3_prot["enrichment_LPS_endfoot_vs_astro"] = (
    stat3_prot["mean_endfoot_LPS"] - stat3_prot["mean_astro_LPS"]
)

stat3_prot["enrichment_all_endfoot_vs_astro"] = (
    stat3_prot["mean_endfoot_all"] - stat3_prot["mean_astro_all"]
)


# =========================
# MODERATED TWO-SIDED t-TEST
# empirical Bayes approximation
# =========================
def moderated_ttest(group1, group2, global_var, prior_df=4):

    x1 = np.array(group1, dtype=float)
    x2 = np.array(group2, dtype=float)

    x1 = x1[~np.isnan(x1)]
    x2 = x2[~np.isnan(x2)]

    if len(x1) < 2 or len(x2) < 2:
        return np.nan

    n1 = len(x1)
    n2 = len(x2)

    mean1 = np.mean(x1)
    mean2 = np.mean(x2)

    var1 = np.var(x1, ddof=1)
    var2 = np.var(x2, ddof=1)

    pooled_var = (
        ((n1 - 1) * var1)
        + ((n2 - 1) * var2)
    ) / (n1 + n2 - 2)

    moderated_var = (
        prior_df * global_var
        + (n1 + n2 - 2) * pooled_var
    ) / (prior_df + n1 + n2 - 2)

    se = np.sqrt(
        moderated_var * (1 / n1 + 1 / n2)
    )

    t_stat = (mean1 - mean2) / se
    df = prior_df + n1 + n2 - 2

    pval = 2 * (1 - t.cdf(abs(t_stat), df))

    return pval


# =========================
# GLOBAL VARIANCE
# =========================
global_var = np.nanmedian([
    np.nanvar(stat3_prot[c], ddof=1)
    for c in all_sample_cols
])

astro_lps_vs_pbs_pvals = []
endfoot_lps_vs_pbs_pvals = []
pbs_astro_vs_endfoot_pvals = []
lps_astro_vs_endfoot_pvals = []

for _, row in stat3_prot.iterrows():

    astro_pbs = row[astro_pbs_cols].astype(float).values
    astro_lps = row[astro_lps_cols].astype(float).values

    endfoot_pbs = row[endfoot_pbs_cols].astype(float).values
    endfoot_lps = row[endfoot_lps_cols].astype(float).values

    p1 = moderated_ttest(
        astro_lps,
        astro_pbs,
        global_var
    )

    p2 = moderated_ttest(
        endfoot_lps,
        endfoot_pbs,
        global_var
    )

    p3 = moderated_ttest(
        endfoot_pbs,
        astro_pbs,
        global_var
    )

    p4 = moderated_ttest(
        endfoot_lps,
        astro_lps,
        global_var
    )

    astro_lps_vs_pbs_pvals.append(p1)
    endfoot_lps_vs_pbs_pvals.append(p2)
    pbs_astro_vs_endfoot_pvals.append(p3)
    lps_astro_vs_endfoot_pvals.append(p4)

stat3_prot["p_astro_LPS_vs_PBS"] = astro_lps_vs_pbs_pvals
stat3_prot["p_endfoot_LPS_vs_PBS"] = endfoot_lps_vs_pbs_pvals
stat3_prot["p_astro_vs_endfoot_PBS"] = pbs_astro_vs_endfoot_pvals
stat3_prot["p_astro_vs_endfoot_LPS"] = lps_astro_vs_endfoot_pvals

# =========================
# FDR CORRECTION
# =========================
stat3_prot["FDR_astro_LPS_vs_PBS"] = benjamini_hochberg(
    stat3_prot["p_astro_LPS_vs_PBS"].fillna(1).values
)

stat3_prot["FDR_endfoot_LPS_vs_PBS"] = benjamini_hochberg(
    stat3_prot["p_endfoot_LPS_vs_PBS"].fillna(1).values
)

stat3_prot["FDR_astro_vs_endfoot_LPS"] = benjamini_hochberg(
    stat3_prot["p_astro_vs_endfoot_LPS"].fillna(1).values
)

stat3_prot["FDR_astro_vs_endfoot_PBS"] = benjamini_hochberg(
    stat3_prot["p_astro_vs_endfoot_PBS"].fillna(1).values
)

# =========================
# SAVE STATS TABLE
# =========================
heatmap_stats_table = stat3_prot[[
    "Genes_first",

    "mean_astro_PBS",
    "mean_astro_LPS",
    "mean_endfoot_PBS",
    "mean_endfoot_LPS",

    "p_astro_LPS_vs_PBS",
    "FDR_astro_LPS_vs_PBS",

    "p_endfoot_LPS_vs_PBS",
    "FDR_endfoot_LPS_vs_PBS",

    "p_astro_vs_endfoot_LPS",
    "FDR_astro_vs_endfoot_LPS",

    "p_astro_vs_endfoot_PBS",
    "FDR_astro_vs_endfoot_PBS"
]].copy()

heatmap_stats_table.to_excel(
    OUT_HEATMAP_STATS,
    index=False
)

# =========================
# SORT TABLE
# =========================
stat3_prot = stat3_prot.sort_values(
    "enrichment_all_endfoot_vs_astro",
    ascending=False
)

stat3_prot.to_excel(
    OUT_TABLE,
    index=False
)

# =========================
# COLORS
# =========================
VEH_COLOR = "#5B8FD9"   # darker blue
LPS_COLOR = "#FF0000"   # true red

# =========================
# MERGED PBS/LPS BARPLOT
# =========================
bar_merged = stat3_prot[[
    "Genes_first",
    "enrichment_PBS_endfoot_vs_astro",
    "enrichment_LPS_endfoot_vs_astro"
]].copy()

bar_merged = bar_merged.dropna(
    subset=[
        "enrichment_PBS_endfoot_vs_astro",
        "enrichment_LPS_endfoot_vs_astro"
    ],
    how="all"
)

x = np.arange(len(bar_merged))
width = 0.4

plt.figure(figsize=(max(12, len(bar_merged) * 0.45), 6))

plt.bar(
    x - width / 2,
    bar_merged["enrichment_PBS_endfoot_vs_astro"],
    width,
    label="VEH",
    color=VEH_COLOR
)

plt.bar(
    x + width / 2,
    bar_merged["enrichment_LPS_endfoot_vs_astro"],
    width,
    label="LPS",
    color=LPS_COLOR
)

plt.axhline(0, linestyle="--", linewidth=1)

plt.xticks(
    x,
    bar_merged["Genes_first"],
    rotation=90
)

plt.ylabel("log2 enrichment (PvAP - astrocyte)")
plt.title(
    "STAT3-relevant proteins: PvAP vs astrocyte enrichment in VEH and LPS"
)
plt.legend()

plt.tight_layout()
plt.savefig(
    OUT_BARPLOT_PBS_LPS_MERGED,
    dpi=300
)
plt.close()

# =========================
# STAT3 ONLY PLOT
# =========================
stat3_only = stat3_prot[
    stat3_prot["Genes_first"].str.upper() == "STAT3"
]

if len(stat3_only) > 0:

    stat3_row = stat3_only.iloc[0]

    stat3_veh = stat3_row[
        "enrichment_PBS_endfoot_vs_astro"
    ]

    stat3_lps = stat3_row[
        "enrichment_LPS_endfoot_vs_astro"
    ]

    plt.figure(figsize=(4, 5))

    plt.bar(
        ["VEH", "LPS"],
        [stat3_veh, stat3_lps],
        color=[VEH_COLOR, LPS_COLOR]
    )

    plt.axhline(
        0,
        linestyle="--",
        linewidth=1,
        color="black"
    )

    plt.ylabel(
        "log2 enrichment (PvAP - astrocyte)"
    )

    plt.title(
        "STAT3 enrichment"
    )

    plt.tight_layout()

    plt.savefig(
        OUT_BARPLOT_STAT3_ONLY,
        dpi=300
    )

    plt.close()

# =========================
# PPTX
# =========================
def add_image_slide(prs, image_path, title):

    slide = prs.slides.add_slide(
        prs.slide_layouts[6]
    )

    title_box = slide.shapes.add_textbox(
        Inches(0.4),
        Inches(0.2),
        Inches(12.5),
        Inches(0.4)
    )

    title_box.text_frame.text = title

    slide.shapes.add_picture(
        image_path,
        Inches(0.4),
        Inches(0.75),
        width=Inches(12.5)
    )


def add_editable_stat3_chart(
    prs,
    stat3_veh,
    stat3_lps
):

    slide = prs.slides.add_slide(
        prs.slide_layouts[6]
    )

    chart_data = CategoryChartData()
    chart_data.categories = [
        "VEH",
        "LPS"
    ]

    chart_data.add_series(
        "STAT3 enrichment",
        [stat3_veh, stat3_lps]
    )

    chart = slide.shapes.add_chart(
        XL_CHART_TYPE.COLUMN_CLUSTERED,
        Inches(3.5),
        Inches(1.5),
        Inches(5),
        Inches(4),
        chart_data
    ).chart

    chart.has_legend = False

    series = chart.series[0]

    point0 = series.points[0]
    point0.format.fill.solid()
    point0.format.fill.fore_color.rgb = RGBColor(
        91, 143, 217
    )

    point1 = series.points[1]
    point1.format.fill.solid()
    point1.format.fill.fore_color.rgb = RGBColor(
        255, 0, 0
    )

prs = Presentation()
prs.slide_width = Inches(13.33)
prs.slide_height = Inches(7.5)

add_image_slide(
    prs,
    OUT_BARPLOT_PBS_LPS_MERGED,
    "STAT3-relevant proteins: PvAP vs astrocyte enrichment in VEH and LPS"
)

add_image_slide(
    prs,
    OUT_BARPLOT_STAT3_ONLY,
    "STAT3 enrichment"
)

if len(stat3_only) > 0:

    add_editable_stat3_chart(
        prs,
        stat3_veh,
        stat3_lps
    )

prs.save(OUT_PPTX)

print("\nSaved:")
print(OUT_PPTX)
print(OUT_BARPLOT_STAT3_ONLY)
print(OUT_BARPLOT_PBS_LPS_MERGED)
print(OUT_HEATMAP_STATS)